# URL FOR BASIC DATA ON ALL POKEMON FROM ALL GENERATIONS

# GPU ACCELERATION

In [1]:
%load_ext cudf.pandas
%load_ext cuml.accel

# IMPORTS

In [2]:
import os
import io
import re
import time
import urllib.request
import urllib.error
import pandas as pd
from bs4 import BeautifulSoup

BASE_URL = 'https://bulbapedia.bulbagarden.net'
DATA_DIR = os.path.expanduser('~/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x465-003/data/')
_HEADERS = {'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64; rv:132.0) Gecko/20100101 Firefox/132.0'}

In [3]:
pokemon_ALL_url = "https://bulbapedia.bulbagarden.net/wiki/List_of_Pok%C3%A9mon_by_base_stats_in_Generation_IX"

# SCRAPER

In [4]:
def scrape_pokemon_base_stats(url):
    """
    Scrapes the Bulbapedia base stats table using direct HTTP (no headless browser needed
    since the table is server-rendered HTML).
    Table columns: #, Pokémon (sprite), Pokémon.1 (name), HP, Attack, Defense,
                   Sp. Atk, Sp. Def, Speed, Total, Average
    Returns a list of dicts matching the first 10 columns of pokemon_dataset_MASTER.csv.
    """
    req = urllib.request.Request(url, headers={
        'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64; rv:132.0) Gecko/20100101 Firefox/132.0'
    })
    with urllib.request.urlopen(req, timeout=30) as resp:
        html = resp.read().decode('utf-8')

    dfs = pd.read_html(io.StringIO(html))
    df = next(
        t for t in dfs
        if 'HP' in t.columns and 'Attack' in t.columns
    )

    records = []
    for _, row in df.iterrows():
        try:
            pokedex_num = int(row['#'])
            name        = str(row['Pokémon.1']).strip()   # .1 suffix = name col; plain 'Pokémon' is the sprite (NaN)
            hp          = int(row['HP'])
            attack      = int(row['Attack'])
            defense     = int(row['Defense'])
            sp_atk      = int(row['Sp. Atk'])
            sp_def      = int(row['Sp. Def'])
            speed       = int(row['Speed'])
            total       = int(row['Total'])
            link = f"{BASE_URL}/wiki/{name.replace(' ', '_')}"
        except (ValueError, TypeError):
            continue

        records.append({
            'Pokedex Number': pokedex_num,
            'Pokemon':        name,
            'Link':           link,
            'HP':             hp,
            'Attack':         attack,
            'Defense':        defense,
            'Speed':          speed,
            'Special Attack': sp_atk,
            'Special Defense': sp_def,
            'Stat Total':     total,
        })

    return records


records = scrape_pokemon_base_stats(pokemon_ALL_url)
pokes_scraped = pd.DataFrame(records)
print(f"Scraped {len(pokes_scraped)} rows.")
pokes_scraped.head(10)

Scraped 1239 rows.


,Pokedex Number,Pokemon,Link,HP,Attack,Defense,Speed,Special Attack,Special Defense,Stat Total
0,1,Bulbasaur,https://bulbapedia.bulbagarden.net/wiki/Bulbasaur,45,49,49,45,65,65,318
1,2,Ivysaur,https://bulbapedia.bulbagarden.net/wiki/Ivysaur,60,62,63,60,80,80,405
2,3,Venusaur,https://bulbapedia.bulbagarden.net/wiki/Venusaur,80,82,83,80,100,100,525
3,3,Venusaur Mega Venusaur,https://bulbapedia.bulbagarden.net/wiki/Venusa...,80,100,123,80,122,120,625
4,4,Charmander,https://bulbapedia.bulbagarden.net/wiki/Charma...,39,52,43,65,60,50,309
5,5,Charmeleon,https://bulbapedia.bulbagarden.net/wiki/Charme...,58,64,58,80,80,65,405
6,6,Charizard,https://bulbapedia.bulbagarden.net/wiki/Charizard,78,84,78,100,109,85,534
7,6,Charizard Mega Charizard X,https://bulbapedia.bulbagarden.net/wiki/Chariz...,78,130,111,100,130,85,634
8,6,Charizard Mega Charizard Y,https://bulbapedia.bulbagarden.net/wiki/Chariz...,78,104,78,100,159,115,634
9,7,Squirtle,https://bulbapedia.bulbagarden.net/wiki/Squirtle,44,48,65,43,50,64,314


# VARIATION EXTRACTION (Column 10)
Regex-extracted variation name from the Pokemon name column.
Based on the pattern from the original project (`pokesproject_main.ipynb`).

In [5]:
_VAR_PATTERN = re.compile(
    r'Mega.*[A-Za-z]|Alolan.*[A-Za-z]|Partner.*[A-Za-z]|Galarian.*[A-Za-z]'
    r'|Hisuian.*[A-Za-z]|Paldean.*[A-Za-z]+\D|in L.*[A-Za-z]|Primal.*[A-Za-z]'
    r'|[A-Za-z]+.Forme|[A-Za-z]+.Cloak|[A-Za-z]+.Rotom|[A-Za-z]+.Mode'
    r'|[A-Za-z]+.Kyurem|Ash.*[A-Za-z]|Eternal.*[A-Za-z]|[A-Za-z]+.Size'
    r'|[A-Za-z0-9]+%.Forme|Hoopa(?![\s\S]*?Hoopa).*[A-Za-z]'
    r'|D.*[A-Za-z]+.*Necrozma|[A-Za-z]+.Necrozma|Hero.*[A-Za-z]'
    r'|Crowned.*[A-Za-z]|Eternamax.*[A-Za-z]|\b\w+\s+Rider.*[A-Za-z]'
    r'|Bloodmoon.*[A-Za-z]|Male|Female'
)

variations = []
for name in pokes_scraped['Pokemon']:
    match = _VAR_PATTERN.search(name)
    variations.append(match.group() if match else '')

pokes_scraped.insert(10, 'Variation', variations)

n_var = sum(1 for v in variations if v != '')
print(f"Variation column added. {n_var} variants found out of {len(pokes_scraped)} rows.")
pokes_scraped[pokes_scraped['Variation'] != ''].head(10)

Variation column added. 213 variants found out of 1239 rows.


,Pokedex Number,Pokemon,Link,HP,Attack,Defense,Speed,Special Attack,Special Defense,Stat Total,Variation
3,3,Venusaur Mega Venusaur,https://bulbapedia.bulbagarden.net/wiki/Venusa...,80,100,123,80,122,120,625,Mega Venusaur
7,6,Charizard Mega Charizard X,https://bulbapedia.bulbagarden.net/wiki/Chariz...,78,130,111,100,130,85,634,Mega Charizard X
8,6,Charizard Mega Charizard Y,https://bulbapedia.bulbagarden.net/wiki/Chariz...,78,104,78,100,159,115,634,Mega Charizard Y
12,9,Blastoise Mega Blastoise,https://bulbapedia.bulbagarden.net/wiki/Blasto...,79,103,120,78,135,115,630,Mega Blastoise
19,15,Beedrill Mega Beedrill,https://bulbapedia.bulbagarden.net/wiki/Beedri...,65,150,40,145,15,80,495,Mega Beedrill
23,18,Pidgeot Mega Pidgeot,https://bulbapedia.bulbagarden.net/wiki/Pidgeo...,83,80,80,121,135,80,579,Mega Pidgeot
25,19,Rattata Alolan Form,https://bulbapedia.bulbagarden.net/wiki/Rattat...,30,56,35,72,25,35,253,Alolan Form
27,20,Raticate Alolan Form,https://bulbapedia.bulbagarden.net/wiki/Ratica...,75,71,70,77,40,80,413,Alolan Form
33,25,Pikachu Partner Pikachu,https://bulbapedia.bulbagarden.net/wiki/Pikach...,45,80,50,120,75,60,430,Partner Pikachu
35,26,Raichu Alolan Form,https://bulbapedia.bulbagarden.net/wiki/Raichu...,60,85,50,110,95,85,485,Alolan Form


# INFOBOX SCRAPER (Category, Type 1, Type 2, Height, Weight)
Visits each Pokemon's individual Bulbapedia page and extracts data from the
right-side infobox (`<table class="roundy infobox">`). Variant Pokemon use the
Variation column to select the correct form's data from multi-form infoboxes.

In [6]:
_TYPE_HREF = re.compile(r'/wiki/\w+_\(type\)')
_page_cache = {}


def fetch_page(url):
    """Cached HTTP fetch -> BeautifulSoup. Rate-limits 0.3s on cache miss."""
    if url in _page_cache:
        return _page_cache[url]
    req = urllib.request.Request(url, headers=_HEADERS)
    time.sleep(0.3)
    with urllib.request.urlopen(req, timeout=30) as resp:
        html = resp.read().decode('utf-8')
    soup = BeautifulSoup(html, 'html.parser')
    _page_cache[url] = soup
    return soup


def get_infobox(soup):
    """Find the main infobox: <table class='roundy infobox'>."""
    for table in soup.find_all('table'):
        cls = table.get('class', [])
        if 'roundy' in cls and 'infobox' in cls:
            return table
    return None


def parse_category(infobox):
    """Extract species category from <a href='/wiki/Pokémon_category'> -> <span>."""
    if not infobox:
        return ''
    link = infobox.find('a', href='/wiki/Pok%C3%A9mon_category')
    if link is None:
        link = infobox.find('a', href=lambda h: h and 'category' in h.lower() and 'mon' in h.lower())
    if link:
        span = link.find('span')
        text = span.get_text(strip=True) if span else link.get_text(strip=True)
        return text.split('\n')[0]
    return ''


def parse_types(infobox, variation):
    """
    Extract (Type 1, Type 2) from the infobox.
    Multi-form: each form's types are in a separate <td> with a <small> form label.
    Hidden unused forms have style='display: none' on their <td>.
    """
    if not infobox:
        return ('', '')

    # Collect visible <td> elements containing type links
    type_tds = []
    for td in infobox.find_all('td'):
        links = td.find_all('a', href=_TYPE_HREF)
        if not links:
            continue
        style = td.get('style', '')
        if 'display: none' in style or 'display:none' in style:
            continue
        type_tds.append(td)

    if not type_tds:
        return ('', '')

    # Pick the right <td> for the requested form
    target = None
    if variation:
        var_clean = variation.replace('\xa0', ' ')
        # Exact match on <small> form label
        for td in type_tds:
            small = td.find('small')
            if small:
                label = small.get_text(strip=True).replace('\xa0', ' ')
                if label == var_clean:
                    target = td
                    break
        # Substring fallback
        if target is None:
            for td in type_tds:
                small = td.find('small')
                if small:
                    label = small.get_text(strip=True).replace('\xa0', ' ')
                    if var_clean in label or label in var_clean:
                        target = td
                        break

    if target is None:
        target = type_tds[0]

    # Extract type names from the target <td>
    seen = []
    for a in target.find_all('a', href=_TYPE_HREF):
        b = a.find('b')
        name = b.get_text(strip=True) if b else ''
        if not name:
            m = re.search(r'/wiki/(\w+)_\(type\)', a.get('href', ''))
            name = m.group(1) if m else ''
        if name and name not in seen:
            seen.append(name)

    return (seen[0] if seen else '', seen[1] if len(seen) > 1 else '')


def parse_hw(infobox, variation, field='Height'):
    """
    Extract Height (m) or Weight (kg) from the infobox.
    Sub-table after <b>Height</b>/<b>Weight</b> has repeating row pairs:
      [value row: imperial | metric], [name row: <small>FormName</small>]
    All form rows are visible; only unused slots have display:none.
    """
    if not infobox:
        return ''

    b_tag = None
    for b in infobox.find_all('b'):
        if b.get_text(strip=True) == field:
            b_tag = b
            break
    if not b_tag:
        return ''

    container = b_tag.find_parent('td')
    if not container:
        return ''
    sub_table = container.find('table')
    if not sub_table:
        return ''

    rows = sub_table.find_all('tr')

    # Build (metric_value, form_name, is_hidden) triples from row pairs
    pairs = []
    i = 0
    while i < len(rows):
        tds = rows[i].find_all('td', recursive=False)
        if len(tds) >= 2:
            # Value row -> metric is in the second <td>
            metric_text = tds[1].get_text(strip=True)
            metric_val = re.sub(r'[^0-9.+]', '', metric_text)

            form_name = ''
            hidden = 'display:none' in rows[i].get('style', '').replace(' ', '')
            if i + 1 < len(rows):
                small = rows[i + 1].find('small')
                if small:
                    form_name = small.get_text(strip=True).replace('\xa0', ' ')
                    hidden = hidden or 'display:none' in rows[i + 1].get('style', '').replace(' ', '')
                    i += 2
                else:
                    i += 1
            else:
                i += 1

            try:
                val = float(metric_val)
            except ValueError:
                val = ''

            pairs.append((val, form_name, hidden))
        else:
            i += 1

    if not pairs:
        return ''

    # Match variation
    if variation:
        var_clean = variation.replace('\xa0', ' ')
        for val, name, _ in pairs:
            if name == var_clean:
                return val
        for val, name, _ in pairs:
            if var_clean in name or name in var_clean:
                return val

    # Base form: first non-hidden pair
    for val, _, hidden in pairs:
        if not hidden:
            return val

    return pairs[0][0]


def resolve_url(pokemon_name, variation, link):
    """
    For variants, strip the variation from the name to get the base Pokemon URL.
    E.g. 'Venusaur Mega Venusaur' with variation 'Mega Venusaur' -> /wiki/Venusaur
    This avoids relying on HTTP redirects and improves cache hits.
    """
    if variation:
        base = re.sub(_VAR_PATTERN, '', pokemon_name).strip().replace(' ', '_')
        if base:
            return f"{BASE_URL}/wiki/{base}"
    return link


def scrape_infobox_row(pokemon_name, variation, link):
    """Scrape one row's infobox data with double-try URL fallback."""
    url = resolve_url(pokemon_name, variation, link)
    try:
        soup = fetch_page(url)
    except Exception:
        try:
            soup = fetch_page(link)
        except Exception:
            return {'Category': '', 'Type 1': '', 'Type 2': '',
                    'Height (m)': '', 'Weight (kg)': ''}

    box = get_infobox(soup)
    t1, t2 = parse_types(box, variation)
    return {
        'Category':    parse_category(box),
        'Type 1':      t1,
        'Type 2':      t2,
        'Height (m)':  parse_hw(box, variation, 'Height'),
        'Weight (kg)': parse_hw(box, variation, 'Weight'),
    }


print("Infobox helper functions defined.")

Infobox helper functions defined.


In [7]:
# --- Main infobox scraping loop ---
categories, types1, types2, heights, weights = [], [], [], [], []

total = len(pokes_scraped)
print(f"Scraping infobox data for {total} rows...")

for i, row in pokes_scraped.iterrows():
    name = row['Pokemon']
    variation = row['Variation']
    link = row['Link']

    data = scrape_infobox_row(name, variation, link)
    categories.append(data['Category'])
    types1.append(data['Type 1'])
    types2.append(data['Type 2'])
    heights.append(data['Height (m)'])
    weights.append(data['Weight (kg)'])

    if (i + 1) % 50 == 0 or i == total - 1:
        print(f"  [{i + 1}/{total}] {name}")

pokes_scraped['Category'] = categories
pokes_scraped['Type 1'] = types1
pokes_scraped['Type 2'] = types2
pokes_scraped['Height (m)'] = heights
pokes_scraped['Weight (kg)'] = weights

print(f"\nDone. {len(_page_cache)} unique pages fetched.")
print(f"Columns: {list(pokes_scraped.columns)}")
pokes_scraped.head(10)

Scraping infobox data for 1239 rows...
  [50/1239] Clefable
  [100/1239] Geodude
  [150/1239] Lickitung
  [200/1239] Zapdos Galarian Form
  [250/1239] Hoppip
  [300/1239] Houndour
  [350/1239] Lombre
  [400/1239] Plusle
  [450/1239] Tropius
  [500/1239] Piplup
  [550/1239] Mime Jr.
  [600/1239] Rotom Frost Rotom
  [650/1239] Pidove
  [700/1239] Scrafty Mega Scrafty
  [750/1239] Beheeyem
  [800/1239] Keldeo
  [850/1239] Malamar
  [900/1239] Hoopa Hoopa Confined
  [950/1239] Comfey
  [1000/1239] Zeraora
  [1050/1239] Hatenna
  [1100/1239] Calyrex Shadow Rider
  [1150/1239] Shroodle
  [1200/1239] Iron Treads
  [1239/1239] Pecharunt

Done. 1015 unique pages fetched.
Columns: ['Pokedex Number', 'Pokemon', 'Link', 'HP', 'Attack', 'Defense', 'Speed', 'Special Attack', 'Special Defense', 'Stat Total', 'Variation', 'Category', 'Type 1', 'Type 2', 'Height (m)', 'Weight (kg)']


,Pokedex Number,Pokemon,Link,HP,Attack,Defense,Speed,Special Attack,Special Defense,Stat Total,Variation,Category,Type 1,Type 2,Height (m),Weight (kg)
0,1,Bulbasaur,https://bulbapedia.bulbagarden.net/wiki/Bulbasaur,45,49,49,45,65,65,318,,Seed Pokémon,Grass,Poison,0.7,6.9
1,2,Ivysaur,https://bulbapedia.bulbagarden.net/wiki/Ivysaur,60,62,63,60,80,80,405,,Seed Pokémon,Grass,Poison,1.0,13.0
2,3,Venusaur,https://bulbapedia.bulbagarden.net/wiki/Venusaur,80,82,83,80,100,100,525,,Seed Pokémon,Grass,Poison,2.0,100.0
3,3,Venusaur Mega Venusaur,https://bulbapedia.bulbagarden.net/wiki/Venusa...,80,100,123,80,122,120,625,Mega Venusaur,Seed Pokémon,Grass,Poison,2.4,155.5
4,4,Charmander,https://bulbapedia.bulbagarden.net/wiki/Charma...,39,52,43,65,60,50,309,,Lizard Pokémon,Fire,Unknown,0.6,8.5
5,5,Charmeleon,https://bulbapedia.bulbagarden.net/wiki/Charme...,58,64,58,80,80,65,405,,Flame Pokémon,Fire,Unknown,1.1,19.0
6,6,Charizard,https://bulbapedia.bulbagarden.net/wiki/Charizard,78,84,78,100,109,85,534,,Flame Pokémon,Fire,Flying,1.7,90.5
7,6,Charizard Mega Charizard X,https://bulbapedia.bulbagarden.net/wiki/Chariz...,78,130,111,100,130,85,634,Mega Charizard X,Flame Pokémon,Fire,Dragon,1.7,110.5
8,6,Charizard Mega Charizard Y,https://bulbapedia.bulbagarden.net/wiki/Chariz...,78,104,78,100,159,115,634,Mega Charizard Y,Flame Pokémon,Fire,Flying,1.7,100.5
9,7,Squirtle,https://bulbapedia.bulbagarden.net/wiki/Squirtle,44,48,65,43,50,64,314,,Tiny TurtlePokémon,Water,Unknown,0.5,9.0


In [8]:
# --- Validation spot-check against known values ---
_checks = [
    ('Bulbasaur',      '', {'Category': 'Seed Pokémon',   'Type 1': 'Grass',    'Type 2': 'Poison',  'Height (m)': 0.7,  'Weight (kg)': 6.9}),
    ('Venusaur Mega Venusaur', 'Mega Venusaur', {'Category': 'Seed Pokémon', 'Type 1': 'Grass', 'Type 2': 'Poison', 'Height (m)': 2.4, 'Weight (kg)': 155.5}),
    ('Charmander',     '', {'Category': 'Lizard Pokémon', 'Type 1': 'Fire',     'Type 2': '',        'Height (m)': 0.6,  'Weight (kg)': 8.5}),
    ('Raichu Alolan Raichu', 'Alolan Raichu', {'Category': 'Mouse Pokémon', 'Type 1': 'Electric', 'Type 2': 'Psychic', 'Height (m)': 0.7, 'Weight (kg)': 21.0}),
]

all_ok = True
for poke_name, var, expected in _checks:
    mask = pokes_scraped['Pokemon'] == poke_name
    if mask.sum() == 0:
        print(f"[SKIP] {poke_name} not found in scraped data")
        continue
    row = pokes_scraped.loc[mask].iloc[0]
    errors = []
    for col, exp_val in expected.items():
        got = row[col]
        # Normalize for comparison
        if isinstance(exp_val, float):
            try:
                match = abs(float(got) - exp_val) < 0.05
            except (ValueError, TypeError):
                match = False
        else:
            match = str(got).strip() == str(exp_val).strip()
        if not match:
            errors.append(f"  {col}: expected={exp_val!r}, got={got!r}")
    if errors:
        print(f"[FAIL] {poke_name}")
        for e in errors:
            print(e)
        all_ok = False
    else:
        print(f"[OK]   {poke_name}")

if all_ok:
    print("\nAll spot-checks passed!")
else:
    print("\nSome checks failed — review the output above.")

[OK]   Bulbasaur
[OK]   Venusaur Mega Venusaur
[FAIL] Charmander
  Type 2: expected='', got='Unknown'
[SKIP] Raichu Alolan Raichu not found in scraped data

Some checks failed — review the output above.


# SAVE IT

In [9]:
pokes_scraped.to_csv(DATA_DIR + 'pokemon_base_stats_scraped.csv', index=False)
print(f"Saved {len(pokes_scraped)} rows to {DATA_DIR}pokemon_base_stats_scraped.csv")

Saved 1239 rows to /home/wanderduck/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x465-003/data/pokemon_base_stats_scraped.csv


In [12]:
pokes_scraped['Type 2'] = pokes_scraped['Type 2'].replace('Unknown', None)

In [13]:
pokes_scraped.to_csv(DATA_DIR + 'pokemon_base_stats_scraped.csv', index=False)